# RSNA Knee — bigger LLMs and calibration on the 58 true labels
上位は0.96台。画像モデルは教わったラベルの質にほぼ1対1でついていくので、ラベルを作るLLMの質が鍵。より大きなLLMを58件で試し、正解58件で出力を補正(交差検証)した効果も見る。

In [ ]:
import os, sys, subprocess, warnings
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # メモリの断片化によるOOMを減らす
os.environ['PYTHONWARNINGS'] = 'ignore'  # joblibのワーカープロセスにも効かせる
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
_name = 'rsna-knee-abnormality-detection'
_cands = [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name]
COMP = next((d for d in _cands if (d/'train.csv').exists()), None)
assert COMP, f"train.csv が見つからない。/kaggle/input: {[p.name for p in Path('/kaggle/input').iterdir()]}"
print('COMP =', COMP)
REPO = Path('/kaggle/temp/repo') if Path('/kaggle/temp').exists() else Path('/tmp/repo')   # /kaggle/working に置くと実行の出力に含まれてしまう
SRC = REPO/'rsna-knee-abnormality-detection'/'src'
if not SRC.exists():  # internet ON のときだけ。OFFなら src を Dataset で追加して SRC を変更
    subprocess.run(['git','clone','--depth','1','-b','main',
                    'https://github.com/hidenori24/kaggle_ipynb', str(REPO)], check=True)
sys.path.insert(0, str(SRC))
from pseudo_labels import LABELS
from multilingual_labels import add_pseudo_labels_ml
print('commit:', subprocess.run(['git','-C',str(SRC),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
os.environ['HF_HOME'] = '/kaggle/temp/hf' if Path('/kaggle/temp').exists() else '/tmp/hf'   # モデルのキャッシュは出力に含めない
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes', 'accelerate'], check=False)
import time, torch, transformers
import llm_labels as LL
print('transformers', transformers.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
from multilingual_labels import label_report_lang
from sklearn.metrics import roc_auc_score
train = pd.read_csv(COMP/'train.csv')
train['has_true'] = train['ACL'].notna()
train = add_pseudo_labels_ml(train)
tv = train[train.has_true].reset_index(drop=True)
T = tv[LABELS].values.astype(int)
R = np.array([[label_report_lang(r, l)[c] for c in LABELS] for r, l in zip(tv.Report.fillna(''), tv.lang)])   # 規則ラベル(正解を見ずに作る)
print('true-labeled:', len(tv), tv.lang.value_counts().to_dict())

In [ ]:
import gc, shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODELS = ['Qwen/Qwen2.5-7B-Instruct', 'Qwen/Qwen2.5-14B-Instruct', 'mistralai/Mistral-Nemo-Instruct-2407']   # 7Bは基準(07と同じ)
VARIANTS, BS = ['v0', 'v1'], 8
reports = tv.Report.fillna('').tolist()
def load4(name):
    tok = AutoTokenizer.from_pretrained(name)
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    return tok, AutoModelForCausalLM.from_pretrained(name, device_map='auto', quantization_config=q).eval()
runs = {}            # (model, variant) -> dict(P=soft labels (58,12), sec=seconds per report)
for name in MODELS:
    try:
        t_load = time.time(); tok, model = load4(name)
        print(f'== {name}: loaded in {time.time() - t_load:.0f}s, GPU memory GB', round(sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / 1e9, 1))
        for v in VARIANTS:
            t0 = time.time()
            H, P, npar, Lv = LL.label_reports(model, tok, reports, bs=BS, variant=v, return_levels=True)
            runs[(name, v)] = dict(P=P, Lv=Lv, sec=(time.time() - t0) / len(reports))
            print(f'   [{v}] {runs[(name, v)]["sec"]:.2f} s/report -> all {len(train)} reports ~ {runs[(name, v)]["sec"] * len(train) / 60:.0f} min; <12 keys parsed: {int((npar < 12).sum())}')
    except Exception as e:
        print('!!', name, 'failed:', repr(e)[:300])
    finally:
        try: del model, tok
        except NameError: pass
        gc.collect(); torch.cuda.empty_cache()
        shutil.rmtree(Path(os.environ['HF_HOME']) / 'hub', ignore_errors=True)       # ディスクを空ける

In [ ]:
def auc(S):
    return np.array([roc_auc_score(T[:, j], S[:, j]) if len(np.unique(T[:, j])) > 1 else np.nan for j in range(len(LABELS))])
scores = {'rule(hard)': R.astype(float)}
for (m, v), r in runs.items():
    scores[f'{m.split("/")[1]}:{v}'] = r['P']
names = [k for k in scores if k != 'rule(hard)']
if len(names) > 1:
    scores['mean of all runs'] = np.mean([scores[k] for k in names], axis=0)
tab = pd.DataFrame({k: auc(S) for k, S in scores.items()}, index=LABELS).round(3)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print(tab); print('macro AUC vs TRUE labels:'); print(tab.mean().round(3).sort_values(ascending=False))

In [ ]:
# 正解58件で補正: 所見ごとに、全ての(モデル,版)のスコア(logit)から正解を予測するロジスティック回帰を、繰り返し5分割の交差検証で評価
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
def logit(p): p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))
feats = np.stack([logit(scores[k]) for k in names], axis=-1)              # (58, 12, n_runs)
cv_auc = {}
for j, c in enumerate(LABELS):
    y = T[:, j]
    if y.sum() < 5 or (1 - y).sum() < 5: continue
    res_c = []
    for tr_i, te_i in RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0).split(feats[:, j], y):
        if len(np.unique(y[te_i])) < 2: continue
        m = LogisticRegression(C=0.3, max_iter=1000).fit(feats[tr_i, j], y[tr_i])
        res_c.append(roc_auc_score(y[te_i], m.decision_function(feats[te_i, j])))
    cv_auc[c] = np.mean(res_c)
raw = auc(scores['mean of all runs'] if 'mean of all runs' in scores else scores[names[0]])
out = pd.DataFrame({'raw mean': dict(zip(LABELS, raw)), 'stacked (CV)': cv_auc}).round(3)
print(out); print('macro  raw %.3f | stacked(CV) %.3f' % (out['raw mean'].loc[out.dropna().index].mean(), out['stacked (CV)'].mean()))